# Rotate API keys → push to the VM

**One-shot notebook for rotating Bybit + Velotrade API keys on the trading VM.**

What this does:
1. Mounts your Google Drive and reads your VM SSH private key from `My Drive/ICT_Bot_Secrets/`.
2. Reads your API keys + connection details from Colab Secrets.
3. Generates a fresh `.env` (the canonical file the systemd unit reads).
4. Uploads it to the VM via SSH (atomic write, owner-only permissions).
5. Restarts the trader and Telegram-bot systemd units and verifies they're active.
6. **Sweeps for rogue Python processes** still running pre-restart code (symptom: stale `Pipeline result` lines on Telegram with the legacy `ALLOW_LIVE_TRADING=true is required` reason and no `strategy=` field) and kills them.
7. Audits installed `ict-*` systemd units and **auto-masks any disabled, non-canonical** ones (so a stray `systemctl start` can't revive them). When `systemctl mask` is blocked because the unit lives directly in `/etc/systemd/system/`, the notebook falls back to renaming the file to `.disabled-by-claude` (reversible). Active / enabled unknowns are flagged for manual review — they could be a legitimate add-on.

**How to run:** `Runtime → Run all`. The first cell triggers a one-click "Allow Drive access" dialog the first time you run it in a session — click *Allow*. After that, no further interaction.

**If your SSH key isn't in Drive** (or Drive can't be mounted), the notebook automatically falls back to a file picker that lets you upload the key from your computer.

---

## Trading mode (BUG-039 — operator directive 2026-05-03)

The dry/live toggle is **per-account** in `config/accounts.yaml` (`mode: live | dry_run`), applied via `RiskManager.dry_run`. There is **no** process-level interlock — this notebook does **not** render `MODE`, `DRY_RUN`, or `ALLOW_LIVE_TRADING` into the `.env`. To flip an account, edit `config/accounts.yaml` and let the trader reload, or use Telegram `/accounts dry|live <name>` for a runtime override.

---

**Required Colab Secrets** (set them once via `Tools → Secrets`, toggle *Notebook access* on):

| Name | What it holds |
|---|---|
| `BYBIT_API_KEY_1`     | Bybit API key for account `bybit_1` (turtle_soup) |
| `BYBIT_API_SECRET_1`  | Bybit API secret for account `bybit_1` |
| `BYBIT_API_KEY_2`     | Bybit API key for account `bybit_2` (vwap) |
| `BYBIT_API_SECRET_2`  | Bybit API secret for account `bybit_2` |
| `TELEGRAM_BOT_TOKEN`  | Bot token from @BotFather |
| `TELEGRAM_CHAT_ID`    | Your Telegram chat id |
| `VM_SSH_HOST`         | The VM's hostname or public IP |
| `VM_SSH_USER`         | SSH user on the VM (usually `ubuntu`) |

**Required SSH key file** — preferred location:

`My Drive/ICT_Bot_Secrets/ict-bot-ovm-private.key`

(also accepted in the same folder: `vm_ssh_key`, `id_rsa`, `id_ed25519`, `id_ecdsa` — first match wins). If your file is named something else, set the optional `SSH_KEY_FILE` Colab Secret to its filename. If the notebook can't find it in Drive, it'll pop a file-picker so you can upload from your computer.

**Optional Colab Secrets** (skip if unused):

| Name | What it holds |
|---|---|
| `SSH_KEY_FILE`              | A custom SSH key filename if yours doesn't match the defaults above. |
| `VELOTRADE_API_KEY_1`       | Velotrade DXtrade API key for `prop_velotrade_1`. The account loads as `not_configured` in `/accounts_status` until both `VELOTRADE_API_KEY_1` + `VELOTRADE_API_SECRET_1` are set. |
| `VELOTRADE_API_SECRET_1`    | Velotrade DXtrade API secret matching `VELOTRADE_API_KEY_1`. |
| `VELOTRADE_BASE_URL`        | DXtrade base URL — sandbox vs prod. |
| `NEWS_API_KEY`              | NewsAPI key (only if you've enabled the news layer) |
| `TELEGRAM_CLAUDE_BOT_TOKEN` | Token for the Claude bridge bot. Required only if running `ict-claude-bridge.service`. |
| `ANTHROPIC_API_KEY`         | Anthropic API key for the Claude bridge bot. |
| `CLAUDE_MODEL`              | Model id for the Claude bridge (default `claude-opus-4-7`). |

---

**Velotrade onboarding flow** (when you're ready to take `prop_velotrade_1` live):

1. Add `VELOTRADE_API_KEY_1`, `VELOTRADE_API_SECRET_1`, and `VELOTRADE_BASE_URL` to Colab Secrets.
2. Run all cells. The notebook writes them into `.env` so the trader's systemd unit picks them up on restart.
3. Confirm `/accounts_status` flips `prop_velotrade_1` from `⚙️ Not configured` to `✅ Balance …`.
4. Once the DXtrade SDK contract is filled into `src/units/accounts/dxtrade_client.py`, in `config/accounts.yaml` flip `prop_velotrade_1::mode` from `dry_run` to `live` and add the assigned strategies to its `strategies` list.

Until then `prop_velotrade_1` stays `mode: dry_run` (the SDK contract isn't wired) and the strategies list is empty — both are belt-and-braces gates that keep the account inert. Adding the secrets here is **safe** even before the SDK contract lands; trades won't fire until both gates are cleared.

---

**Security:** Colab Secrets are scoped to your Google account and never appear in the notebook source. The SSH key is read from Drive (or your local upload), copied to a Colab tempdir with `0600` perms only for the duration of the SSH call, then wiped in a `finally` block. This notebook never prints, logs, or commits any secret value.


In [ ]:
# Step 1A: Mount Google Drive FIRST so the auth dialog pops before
# anything else runs. This is intentionally a separate, tiny cell so
# the dialog cannot be missed.
import os

from google.colab import drive

print('⏳ Mounting Google Drive (a one-click "Allow" dialog will pop the first time per session)…')
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'⚠️  drive.mount() raised: {exc}')

# Verify the mount actually succeeded. drive.mount() is supposed to
# block until the auth flow finishes, but in some Colab sessions a
# stale token can let it return early without a fresh consent.
if not os.path.exists('/content/drive/MyDrive'):
    print('Drive not visible after first mount; forcing a re-auth…')
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception as exc:
        print(f'⚠️  force-remount also failed: {exc}')

drive_mounted = os.path.exists('/content/drive/MyDrive')
if drive_mounted:
    print('✅ Drive mounted.')
else:
    print('⚠️  Drive is NOT mounted. The next cell will prompt you to '
          'upload the SSH key from your computer instead.')

In [ ]:
# Step 1B: Locate the SSH key. Drive first; if Drive lookup fails
# (file not present, or Drive not mounted), pop a file-picker so the
# operator can upload it from their computer.
import os
from google.colab import userdata

# ict-bot-ovm-private.key is the first candidate because that's the
# canonical OCI key filename. The other names cover the standard SSH
# defaults so any common key works without renaming.
DEFAULT_SSH_KEY_NAMES = [
    'ict-bot-ovm-private.key',
    'vm_ssh_key', 'id_rsa', 'id_ed25519', 'id_ecdsa',
]
DRIVE_SECRETS_FOLDER = '/content/drive/MyDrive/ICT_Bot_Secrets'
FALLBACK_FOLDER = '/content'

# Optional filename override.
override_name = None
try:
    override_name = userdata.get('SSH_KEY_FILE')
except Exception:
    pass

candidate_names = []
if override_name:
    candidate_names.append(override_name)
candidate_names.extend(DEFAULT_SSH_KEY_NAMES)

# A. Try Drive first.
ssh_key_path = None
key_source = None
if drive_mounted:
    for name in candidate_names:
        path = os.path.join(DRIVE_SECRETS_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            key_source = 'Drive'
            break

# B. Try /content/ (in case the operator already drag-dropped one).
if ssh_key_path is None:
    for name in candidate_names:
        path = os.path.join(FALLBACK_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            key_source = 'Files panel'
            break

# C. Last resort: pop a file picker. This is interactive — the cell
# blocks until the operator clicks "Choose Files" and picks one.
if ssh_key_path is None:
    print('SSH key not found in Drive or /content/.')
    print('Looked for these names in '
          + (f'{DRIVE_SECRETS_FOLDER}/ and ' if drive_mounted else '')
          + f'{FALLBACK_FOLDER}/:')
    for name in candidate_names:
        print(f'  - {name}')
    print()
    print('Falling back to direct upload. Click "Choose Files" below '
          'and pick your VM SSH private key file from your computer.')
    print()
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit(
            'No file uploaded. Run this cell again and pick the key file, '
            'or place it in My Drive/ICT_Bot_Secrets/ as '
            'ict-bot-ovm-private.key and re-run all cells.'
        )
    uploaded_name = next(iter(uploaded.keys()))
    ssh_key_path = os.path.join(FALLBACK_FOLDER, uploaded_name)
    key_source = 'computer upload'

# D. Sanity check: must look like a private key, not a public key.
with open(ssh_key_path, 'r') as f:
    first_line = f.readline().strip()
if not first_line.startswith('-----BEGIN'):
    raise SystemExit(
        f'{ssh_key_path} does not look like a private key (first line: '
        f'\"{first_line[:40]}\"). Expected something like \"-----BEGIN '
        f'OPENSSH PRIVATE KEY-----\". Did you point at your public key '
        f'(.pub) instead of the private one?'
    )
if 'PUBLIC KEY' in first_line:
    raise SystemExit(
        f'{ssh_key_path} is a PUBLIC key. Use the matching PRIVATE key '
        f'(no .pub extension).'
    )

print(f'✅ SSH key located: {ssh_key_path}')
print(f'   Source: {key_source}')
print(f'   Looks like a valid private key.')

In [ ]:
# Step 1C: Load Colab Secrets and validate.
from google.colab import userdata

REQUIRED = [
    'BYBIT_API_KEY_1', 'BYBIT_API_SECRET_1',
    'BYBIT_API_KEY_2', 'BYBIT_API_SECRET_2',
    'TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID',
    'VM_SSH_HOST', 'VM_SSH_USER',
]
OPTIONAL = [
    # Velotrade DXtrade prop firm (prop_velotrade_1). Account loads
    # as 'not_configured' in /accounts_status until both KEY + SECRET
    # are set. VELOTRADE_BASE_URL is the sandbox vs prod toggle; the
    # client falls back to the SDK contract default when unset.
    'VELOTRADE_API_KEY_1', 'VELOTRADE_API_SECRET_1', 'VELOTRADE_BASE_URL',
    'NEWS_API_KEY',
    # Claude Telegram bridge (ict-claude-bridge.service). Optional —
    # the service stays disabled on the VM until the operator enables
    # it. Adding these secrets just makes them available in .env.
    'TELEGRAM_CLAUDE_BOT_TOKEN', 'ANTHROPIC_API_KEY', 'CLAUDE_MODEL',
]

secrets = {}
missing = []
for name in REQUIRED:
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if not v:
        missing.append(name)
    else:
        secrets[name] = v

for name in OPTIONAL:
    try:
        v = userdata.get(name)
        if v:
            secrets[name] = v
    except Exception:
        pass

if missing:
    raise SystemExit(
        'Missing required Colab Secrets:\n  - '
        + '\n  - '.join(missing)
        + '\n\nFix: Tools → Secrets, add each missing secret with exactly that name '
        + '(toggle Notebook access on), then run all again.'
    )

loaded_required = [n for n in REQUIRED if n in secrets]
loaded_optional = [n for n in OPTIONAL if n in secrets]
print(f'Loaded {len(loaded_required)} required + {len(loaded_optional)} optional secrets from Colab.')
print(f'  Required:  {sorted(loaded_required)}')
print(f'  Optional:  {sorted(loaded_optional) or "(none)"}')
print()
print(f'VM target: {secrets["VM_SSH_USER"]}@{secrets["VM_SSH_HOST"]}')


In [ ]:
# Step 2: Build the .env content.
#
# Mirrors scripts/render_env_from_master.py::build_live (the single
# canonical render path post-2026-05-03). The .env carries credentials,
# Telegram tokens, exchange selection, risk caps, and per-account API-
# key env vars. It does NOT carry MODE / DRY_RUN / ALLOW_LIVE_TRADING
# (BUG-039 — per-account `mode: live | dry_run` in config/accounts.yaml
# is the only dry/live toggle in the codebase).
#
# Risk caps are hardcoded here to the production defaults from
# config/master-secrets.template.yaml::risk.live. Edit if you intentionally
# want to change a non-secret default.

PRODUCTION_DEFAULTS = {
    'ENVIRONMENT': 'production',
    'EXCHANGE': 'bybit',
    'BYBIT_TESTNET': 'false',
    'SYMBOL': 'BTCUSDT',
    'TIMEFRAME': '1m',
    'DATA_DIR': 'data/',
    'MODEL_DIR': 'ml/models/',
    'LOG_DIR': 'logs/',
    'DB_PATH': 'data/trading.db',
    'MAX_POSITION_USD': '50',
    'MAX_DAILY_LOSS_USD': '25',
    'RISK_PER_TRADE': '0.005',
    'MAX_QTY': '0.001',
    'MAX_OPEN_POSITIONS': '1',
    'NEWS_ENABLED': 'false',
    'NEWS_API_KEY': '',
    # Always-on monitor reconciler (BUG-042 PR 3/3, mirrors .env.example).
    # Without this in the rendered env, _reconcile_open_trades() defaults
    # to false → ghost trades pile up in the journal indefinitely
    # (BUG-041 / BUG-042 surface — observed on 2026-05-04 with trade #24).
    'MONITOR_RECONCILE_ENABLED': 'true',
    # Legacy single-account vars (back-compat with code paths that still
    # read the unsuffixed names; mirrors render_env_from_master.py).
    'BYBIT_API_KEY': secrets['BYBIT_API_KEY_1'],
    'BYBIT_API_SECRET': secrets['BYBIT_API_SECRET_1'],
    'BYBIT_BASE_URL': 'https://api.bybit.com',
}

# Per-account credentials (the names accounts.yaml looks up via api_key_env).
PER_ACCOUNT_KEYS = [
    'BYBIT_API_KEY_1', 'BYBIT_API_SECRET_1',
    'BYBIT_API_KEY_2', 'BYBIT_API_SECRET_2',
    # Velotrade phase-2: account loads as 'not_configured' until both
    # KEY + SECRET are present in the env. The dry/live state is
    # already gated separately by config/accounts.yaml `mode` field.
    'VELOTRADE_API_KEY_1', 'VELOTRADE_API_SECRET_1',
]

# Velotrade base URL (sandbox vs prod). Read by velotrade_client_for in
# src/units/accounts/clients.py. Optional — when unset, the client
# falls back to whatever the DXtrade SDK contract specifies as the
# default once src/units/accounts/dxtrade_client.py is filled in.
VELOTRADE_CONFIG_KEYS = ['VELOTRADE_BASE_URL']

TELEGRAM_KEYS = ['TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID']

# Claude bridge env vars (rendered only when the secrets are present;
# the bridge service is disabled by default).
CLAUDE_BRIDGE_KEYS = ['TELEGRAM_CLAUDE_BOT_TOKEN', 'ANTHROPIC_API_KEY', 'CLAUDE_MODEL']
CLAUDE_MODEL_DEFAULT = 'claude-opus-4-7'


def quote_if_needed(val: str) -> str:
    if any(c in val for c in ' \t#$`\'"\\'):
        return '"' + val.replace('\\', '\\\\').replace('"', '\\"') + '"'
    return val


lines = []
for k, v in PRODUCTION_DEFAULTS.items():
    if k == 'NEWS_API_KEY' and 'NEWS_API_KEY' in secrets:
        v = secrets['NEWS_API_KEY']
    if k == 'NEWS_ENABLED' and 'NEWS_API_KEY' in secrets:
        v = 'true'
    lines.append(f'{k}={quote_if_needed(str(v))}')
for k in TELEGRAM_KEYS:
    lines.append(f'{k}={quote_if_needed(secrets[k])}')
for k in PER_ACCOUNT_KEYS:
    if k in secrets:
        lines.append(f'{k}={quote_if_needed(secrets[k])}')

# Velotrade: render config keys (currently just VELOTRADE_BASE_URL)
# only when present so a missing optional secret stays out of .env.
for k in VELOTRADE_CONFIG_KEYS:
    if k in secrets:
        lines.append(f'{k}={quote_if_needed(secrets[k])}')

# Claude bridge: only render when both Telegram bot token and API key
# are present. Render CLAUDE_MODEL with the default if not overridden.
if 'TELEGRAM_CLAUDE_BOT_TOKEN' in secrets and 'ANTHROPIC_API_KEY' in secrets:
    lines.append(f"TELEGRAM_CLAUDE_BOT_TOKEN={quote_if_needed(secrets['TELEGRAM_CLAUDE_BOT_TOKEN'])}")
    lines.append(f"ANTHROPIC_API_KEY={quote_if_needed(secrets['ANTHROPIC_API_KEY'])}")
    lines.append(f"CLAUDE_MODEL={quote_if_needed(secrets.get('CLAUDE_MODEL', CLAUDE_MODEL_DEFAULT))}")

env_content = '\n'.join(lines) + '\n'

# Print only the KEY NAMES, never values. Operator can confirm shape
# without leaking secrets.
key_names = [line.split('=', 1)[0] for line in lines]
print(f'Generated .env with {len(key_names)} variables:')
for name in key_names:
    print(f'  {name}')


In [ ]:
# Step 3: Push the .env file to the VM via SCP, restart the services, verify.
#
# ict-trader-live.service systemd unit reads
# EnvironmentFile=/home/<user>/ict-trading-bot/.env. Post-2026-05-03
# we no longer write a profile-suffixed mirror — the dry/live toggle
# moved to config/accounts.yaml `mode`, so .env is the single canonical
# file the trader reads.
#
# We restart BOTH the trader AND the Telegram bot, because
# /accounts_status runs in the Telegram bot process — restarting only
# the trader leaves the bot reading stale env vars from when it
# started up.
import os
import shutil
import stat
import subprocess
import tempfile

REPO_PATH_ON_VM = '/home/' + secrets['VM_SSH_USER'] + '/ict-trading-bot'
REMOTE_TARGETS = [
    REPO_PATH_ON_VM + '/.env',         # systemd EnvironmentFile reads this
]
SERVICES_TO_RESTART = [
    'ict-trader-live.service',     # the trading loop
    'ict-telegram-bot.service',    # /accounts_status, /balance, etc. — must restart
                                   # too or the bot keeps reading old env vars.
]

# Stage SSH key + .env in a Colab tempdir with 0600 perms — ssh refuses
# key files with broader permissions, and Drive-mounted files don't
# honour chmod (they always look 0644).
tmpdir = tempfile.mkdtemp(prefix='rotate-keys-')
key_path = os.path.join(tmpdir, 'vm_key')
env_path = os.path.join(tmpdir, '.env')

shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)  # 0600

with open(env_path, 'w') as f:
    f.write(env_content)
os.chmod(env_path, stat.S_IRUSR | stat.S_IWUSR)

host = secrets['VM_SSH_HOST']
user = secrets['VM_SSH_USER']
ssh_target = f'{user}@{host}'
ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', 'UserKnownHostsFile=' + os.path.join(tmpdir, 'known_hosts'),
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]


def _redact(text: str) -> str:
    """Strip any secret values that may have leaked into stderr."""
    if not text:
        return text
    for v in secrets.values():
        if v and len(str(v)) > 8:
            text = text.replace(str(v), '<REDACTED>')
    return text


def run_ssh(cmd: str, *, label: str, allow_fail: bool = False):
    full = ['ssh'] + ssh_opts + [ssh_target, cmd]
    proc = subprocess.run(full, capture_output=True, text=True)
    if proc.returncode != 0:
        msg = f'❌ {label} failed (exit {proc.returncode})'
        print(msg)
        print(_redact(proc.stderr or proc.stdout)[:500])
        if not allow_fail:
            raise SystemExit(f'{label} failed')
    return (proc.stdout or '').strip()


def run_scp(local: str, remote: str, *, label: str):
    full = ['scp'] + ssh_opts + [local, f'{ssh_target}:{remote}']
    proc = subprocess.run(full, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print(_redact(proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')


try:
    print(f'⏳ Connecting to {ssh_target} …')
    run_ssh('echo connected', label='SSH connectivity')
    print('✅ SSH OK')

    for remote_final in REMOTE_TARGETS:
        remote_tmp = remote_final + '.tmp'
        print(f'⏳ Uploading {os.path.basename(remote_final)} (atomic) …')
        run_scp(env_path, remote_tmp, label=f'SCP upload {remote_final}')
        run_ssh(f'chmod 600 {remote_tmp} && mv {remote_tmp} {remote_final}',
                label=f'atomic rename {remote_final}')
        print(f'✅ Wrote {remote_final}')

    for svc in SERVICES_TO_RESTART:
        print(f'⏳ Restarting {svc} …')
        # The Telegram bot service may not be installed on every VM
        # (some setups run only the trader). Allow that one to fail
        # softly so the notebook still reports the trader status.
        allow = svc == 'ict-telegram-bot.service'
        run_ssh(f'sudo -n systemctl restart {svc}',
                label=f'restart {svc}', allow_fail=allow)
        state = run_ssh(f'sudo -n systemctl is-active {svc}',
                        label=f'is-active {svc}', allow_fail=allow)
        if state == 'active':
            print(f'✅ {svc}: active')
        elif state:
            print(f'⚠️  {svc}: {state} — check journalctl on the VM.')
        else:
            print(f'⚠️  {svc}: not installed or not running. '
                  f'/accounts_status reads stale env until this service restarts.')

    # ----- Rogue-process sweep + non-canonical unit audit. -----
    # Catches the symptom where journalctl shows pre-#342 'Pipeline
    # result' lines (no `strategy=` field, ALLOW_LIVE_TRADING reason)
    # *after* a clean systemd restart. Cause: a long-running Python
    # interpreter still holds the pre-fix modules in memory — git-sync
    # of source files does not reload imported modules.
    #
    # Rule: a legitimate process belongs to ONE of the canonical
    # systemd units (its cgroup contains the unit name). Anything else
    # running our app entry-points is rogue and gets killed.
    print()
    print('⏳ Auditing python processes for rogue (pre-restart) survivors …')

    CANONICAL_UNITS = {
        'ict-trader-live.service':   'src.main',
        'ict-telegram-bot.service':  'src.bot.telegram_query_bot',
        'ict-claude-bridge.service': 'src.bot.claude_bridge',
    }
    pgrep_pattern = r'src\.main|src\.bot\.telegram_query_bot|src\.bot\.claude_bridge'
    discover = (
        f"pgrep -af '{pgrep_pattern}' | while read pid rest; do "
        f"  cg=$(head -1 /proc/$pid/cgroup 2>/dev/null); "
        f"  et=$(ps -o etime= -p $pid 2>/dev/null | xargs); "
        f"  printf '%s|%s|%s|%s\\n' \"$pid\" \"$et\" \"$cg\" \"$rest\"; "
        f"done"
    )
    output = run_ssh(discover, label='discover python procs', allow_fail=True)

    rogue_pids, canonical_pids = [], []
    for line in (output or '').splitlines():
        if not line.strip():
            continue
        parts = line.split('|', 3)
        if len(parts) != 4:
            continue
        pid, etime, cgroup, cmd = parts
        match = next((u for u in CANONICAL_UNITS if u in cgroup), None)
        if match:
            canonical_pids.append((pid, match, etime, cmd))
        else:
            rogue_pids.append((pid, cgroup or '<no cgroup>', etime, cmd))

    for pid, unit, et, _cmd in canonical_pids:
        print(f'  ✅ pid={pid:>6} etime={et:<10} [{unit}]')
    for pid, cg, et, cmd in rogue_pids:
        print(f'  ❌ pid={pid:>6} etime={et:<10} cgroup={cg}')
        print(f'         {cmd[:90]}')

    if rogue_pids:
        pids_arg = ' '.join(p for p, *_ in rogue_pids)
        print(f'⏳ SIGTERM → {len(rogue_pids)} rogue pid(s) …')
        run_ssh(f'sudo -n kill {pids_arg}',
                label='SIGTERM rogues', allow_fail=True)
        run_ssh('sleep 3', label='grace period', allow_fail=True)
        survivors_raw = run_ssh(
            f'( for p in {pids_arg}; do [ -d /proc/$p ] && echo $p; done; true )',
            label='check survivors', allow_fail=True,
        )
        survivors = [s.strip() for s in (survivors_raw or '').splitlines() if s.strip()]
        if survivors:
            print(f'⚠️  {len(survivors)} survived SIGTERM → SIGKILL …')
            run_ssh(f'sudo -n kill -9 {" ".join(survivors)}',
                    label='SIGKILL survivors', allow_fail=True)
        print('✅ Rogue processes cleared.')
    else:
        print('✅ No rogue processes — only systemd-managed units running our code.')

    # Auto-mask non-canonical disabled ict-* units. The operator
    # directive (2026-05-03): when the notebook kills an old
    # process, it should also hide stale unit-files so they can't
    # be revived by an accidental `systemctl start <name>`. Safe
    # rule: only mask DISABLED units (already inactive at boot).
    # Active / enabled / static / generated / transient / indirect
    # units may be a legitimate operator add-on — flag those for
    # manual review instead.
    print()
    print('⏳ Auditing systemd ict-* units …')
    KNOWN_UNITS = {
        'ict-trader-live.service', 'ict-telegram-bot.service',
        'ict-claude-bridge.service', 'ict-git-sync.service',
        'ict-git-sync.timer', 'ict-heartbeat.service',
        'ict-heartbeat.timer', 'ict-env-check.service',
        'ict-smoke-once.service', 'ict-web-api.service',
    }
    AUTO_MASK_STATES = {'disabled'}
    units_raw = run_ssh(
        "systemctl list-unit-files 'ict-*' --no-pager --no-legend "
        "| awk '{print $1\"|\"$2}'",
        label='list ict-* unit-files', allow_fail=True,
    )
    to_mask, to_review = [], []
    for line in (units_raw or '').splitlines():
        line = line.strip()
        if not line:
            continue
        name, _, state = line.partition('|')
        if not name or name in KNOWN_UNITS:
            continue
        if state in AUTO_MASK_STATES:
            to_mask.append((name, state))
        else:
            to_review.append((name, state))

    if to_mask:
        print(f'⏳ Auto-masking {len(to_mask)} stale disabled unit(s) …')
        for name, state in to_mask:
            print(f'   - {name}  [{state}]')
        # systemctl mask only works when the corresponding
        # /etc/systemd/system/<name> path is FREE — it tries to
        # symlink that path to /dev/null. Units originally
        # installed in /etc/systemd/system/ (not /lib/systemd/system/)
        # already occupy that path as a regular file, so mask
        # fails with 'File ... already exists'. Per-unit fallback:
        # rename the file out of the way so systemd no longer
        # recognises it (any non-.service suffix works); reversible
        # via the inverse mv if the operator wants the unit back.
        masked_names: list[str] = []
        renamed_names: list[str] = []
        errored_names: list[tuple[str, str]] = []
        for name, _state in to_mask:
            full = ['ssh'] + ssh_opts + [
                ssh_target, f'sudo -n systemctl mask {name} 2>&1'
            ]
            proc = subprocess.run(full, capture_output=True, text=True)
            out = (proc.stdout or '') + (proc.stderr or '')
            if proc.returncode == 0:
                masked_names.append(name)
                continue
            if 'already exists' in out:
                # Fallback: file lives directly in /etc/systemd/system/.
                # Rename it; daemon-reload to drop it from the unit set.
                # Idempotent guard via `[ -f ... ]` so re-runs are safe.
                fallback_cmd = (
                    f'( [ -f /etc/systemd/system/{name} ] && '
                    f'  sudo -n mv /etc/systemd/system/{name} '
                    f'  /etc/systemd/system/{name}.disabled-by-claude || true ) '
                    f'&& sudo -n systemctl daemon-reload'
                )
                fb = subprocess.run(
                    ['ssh'] + ssh_opts + [ssh_target, fallback_cmd],
                    capture_output=True, text=True,
                )
                if fb.returncode == 0:
                    renamed_names.append(name)
                else:
                    errored_names.append((name, _redact(
                        (fb.stderr or fb.stdout or '').strip()
                    )[:200]))
                continue
            errored_names.append((name, _redact(out.strip())[:200]))

        if masked_names:
            print(f'✅ Masked {len(masked_names)} unit(s) via '
                  f'`systemctl mask`: {", ".join(masked_names)}.')
        if renamed_names:
            print(f'✅ Renamed {len(renamed_names)} local unit-file(s) '
                  f'to .disabled-by-claude (systemd no longer '
                  f'recognises them as units): '
                  f'{", ".join(renamed_names)}.')
        if errored_names:
            print(f'⚠️  {len(errored_names)} unit(s) could not be '
                  f'masked or renamed:')
            for name, err in errored_names:
                print(f'   - {name}: {err}')
            print('   To handle manually:')
            print('     sudo systemctl disable --now <name>')
            print('     sudo rm /etc/systemd/system/<name>')
            print('     sudo systemctl daemon-reload')
    if to_review:
        print(f'⚠️  {len(to_review)} non-canonical unit(s) need manual review '
              f'(active / enabled / static — could be a legitimate add-on):')
        for name, state in to_review:
            print(f'   - {name}  [{state}]')
        print('   To handle manually:')
        print('     sudo systemctl disable --now <name>')
        print('     sudo systemctl mask <name>')
    if not to_mask and not to_review:
        print('✅ Only known ict-* units installed.')
finally:
    # Wipe the tempdir copy of the key + env. The original in Drive (or
    # /content/) is untouched, so re-running the notebook just works
    # without re-uploading.
    shutil.rmtree(tmpdir, ignore_errors=True)

print('\nDone. Env pushed, services restarted, rogue processes cleared. Open Telegram and run /accounts_status to verify each account shows a real balance, then watch one tick of /status — Pipeline result lines should now all carry strategy=… and never the legacy ALLOW_LIVE_TRADING reason.')


## Verification

After the cells above complete, in Telegram:

1. **`/accounts_status`** — every account should show ✅ with a real USDT balance.
1. **Watch one tick of `Pipeline result` on Telegram** — every line should now carry `strategy=...` and none should carry `reason=ALLOW_LIVE_TRADING=...`. If you still see the legacy form, re-run this notebook (the rogue sweep is idempotent).
2. **`/smoke_test`** — each account should return ✅ `rejected_too_small` (Bybit accepted the request and rejected for size, which proves the keys work).

**Reversibility.** If the auto-mask hid a unit you actually wanted, undo it on the VM:
- If the notebook reported "Masked … via `systemctl mask`":  `sudo systemctl unmask <name>`.
- If the notebook reported "Renamed … to .disabled-by-claude":  `sudo mv /etc/systemd/system/<name>.disabled-by-claude /etc/systemd/system/<name> && sudo systemctl daemon-reload`.

If any account still shows ❌, the new diagnostic message names the exact reason (missing env var, `Bybit error retCode=10003`, network, etc.). See the troubleshooting section in [`docs/operator/colab-key-rotation.md`](https://github.com/benbaichmankass/ict-trading-bot/blob/main/docs/operator/colab-key-rotation.md).

## When to re-run

- You rotated a Bybit API key (update the matching Colab Secret first).
- You changed the VM's SSH key (replace the file in Drive with the new one — same filename — first).
- You added a new account to `config/accounts.yaml`.
- The bot is alerting on `bybit_get_wallet_balance_failed` with `retCode=10003` (key invalid).

## Cleaning up

Nothing to clean. The SSH key in Drive stays where it is (you put it there intentionally). The Colab tempdir copy is wiped in the `finally` block of step 3.